# 🧪 Análise Completa de A/B Test — SaaSify

**Autor:** Victor Luhan · Marketing Analytics Pleno  
**Objetivo:** Avaliar se mudar o headline da landing page do SaaSify de uma comunicação **funcional** para um **benefício emocional** aumenta a taxa de conversão para trial.

| Grupo | Headline |
|---|---|
| **Controle (A)** | *"Software de Agendamento Online"* |
| **Variação (B)** | *"Pare de Atender o Telefone."* |

---
**Ferramentas:** `pandas` · `numpy` · `scipy` · `statsmodels` · `matplotlib`

## 1. Importações e Configurações

Carregamos apenas bibliotecas essenciais e amplamente disponíveis:
- `pandas` + `numpy`: manipulação de dados
- `scipy.stats`: testes estatísticos (z-test de proporções)
- `statsmodels`: poder estatístico e tamanho de amostra
- `matplotlib`: visualizações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import norm
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize
from statsmodels.stats.power import NormalIndPower
import warnings
warnings.filterwarnings('ignore')

# Configurações de estilo dos gráficos
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

# Paleta de cores do relatório
BLUE   = '#1576ac'
PURPLE = '#7c5cfc'
GREEN  = '#00d4aa'
YELLOW = '#f5a623'
BG     = '#1a1a2e'

print('✅ Bibliotecas carregadas com sucesso!')

## 2. Carregamento e Validação dos Dados

Antes de qualquer análise estatística, é obrigatório:
1. Verificar a estrutura do dataset
2. Checar valores nulos (dados faltantes invalidam o teste)
3. Confirmar que não há usuários duplicados (cada usuário deve aparecer **uma única vez**)
4. Verificar o balanceamento entre grupos (split 50/50)

In [ ]:
# Carrega o dataset
df = pd.read_csv('ab_test_data.csv')

print('=== ESTRUTURA DO DATASET ===')
print(f'Linhas: {len(df):,}')
print(f'Colunas: {list(df.columns)}')
print(f'\nPrimeiras linhas:')
df.head()

In [ ]:
print('=== VALIDAÇÃO DE QUALIDADE ===')

# 1. Valores nulos — qualquer nulo significa dado incompleto
nulls = df.isnull().sum()
print(f'\n[Nulos por coluna]')
print(nulls)

# 2. Duplicatas — cada usuário deve ter sido exposto uma única vez
# (caso contrário, o mesmo usuário estaria influenciando o resultado mais de uma vez)
dupes = df['user_id'].duplicated().sum()
print(f'\n[Usuários duplicados]: {dupes}')

# 3. Valores únicos por coluna
print(f'\n[Variantes presentes]: {df["variant"].unique()}')
print(f'[Valores de conversão]: {df["converted"].unique()}')

# 4. Distribuição entre grupos — deve ser próximo de 50/50
print(f'\n[Distribuição entre grupos]:')
print(df['variant'].value_counts(normalize=True).round(4) * 100)

## 3. Análise Exploratória (EDA)

Calculamos as métricas brutas de cada grupo: total de usuários, conversões e taxa de conversão.

**Taxa de Conversão (CR):**
$$CR = \frac{\text{Conversões}}{\text{Visitantes}}$$

In [ ]:
# Agrupamos por variante e calculamos as métricas principais
resultados = df.groupby('variant').agg(
    usuarios   = ('user_id',   'count'),
    conversoes = ('converted', 'sum')
).reset_index()

# Taxa de conversão = conversões / total de usuários no grupo
resultados['tx_conversao'] = resultados['conversoes'] / resultados['usuarios']

# Extraímos os valores para uso nas análises seguintes
n_A  = resultados.loc[resultados['variant']=='A', 'usuarios'].values[0]
n_B  = resultados.loc[resultados['variant']=='B', 'usuarios'].values[0]
cv_A = resultados.loc[resultados['variant']=='A', 'conversoes'].values[0]
cv_B = resultados.loc[resultados['variant']=='B', 'conversoes'].values[0]
cr_A = cv_A / n_A
cr_B = cv_B / n_B

# Métricas de lift
lift_abs  = cr_B - cr_A
lift_pct  = lift_abs / cr_A
lift_raw  = cv_B - cv_A

print('=== MÉTRICAS DO EXPERIMENTO ===')
print(f'\nGrupo A (Controle):')
print(f'  Usuários:        {n_A:,}')
print(f'  Conversões:      {cv_A:,}')
print(f'  Taxa conversão:  {cr_A:.2%}')
print(f'\nGrupo B (Variação):')
print(f'  Usuários:        {n_B:,}')
print(f'  Conversões:      {cv_B:,}')
print(f'  Taxa conversão:  {cr_B:.2%}')
print(f'\n--- LIFT ---')
print(f'  Lift absoluto:   +{lift_abs*100:.2f} pp')
print(f'  Lift percentual: +{lift_pct*100:.2f}%')
print(f'  Trials extras:   +{lift_raw} conversões')

## 4. Visualizações

Criamos quatro gráficos para tornar os resultados acessíveis a stakeholders não-técnicos.

In [ ]:
# --- GRÁFICO 1: Distribuição de Usuários por Grupo ---
# Serve para confirmar visualmente que o split foi balanceado (próximo de 50/50).
# Um split desbalanceado pode enviesar os resultados.

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

grupos = ['Controle (A)', 'Variação (B)']
valores = [n_A, n_B]
cores   = [PURPLE, GREEN]

bars = ax.bar(grupos, valores, color=cores, width=0.5)

for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'{val:,}', ha='center', va='bottom', color='white',
            fontsize=14, fontweight='bold')

ax.set_title('Distribuição de Usuários por Grupo', color='white', fontsize=15, pad=15)
ax.set_ylabel('Número de Usuários', color='white')
ax.set_ylim(0, max(valores) * 1.15)
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_visible(False)

plt.tight_layout()
plt.savefig('fig1_usuarios.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Fig. 1 salva.')

In [ ]:
# --- GRÁFICO 2: Conversões por Grupo ---
# Mostra o volume absoluto de conversões. Importante para stakeholders que
# querem entender a diferença em números concretos, não só em percentual.

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

valores_cv = [cv_A, cv_B]

bars = ax.bar(grupos, valores_cv, color=cores, width=0.5)

for bar, val in zip(bars, valores_cv):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
            f'{val:,}', ha='center', va='bottom', color='white',
            fontsize=14, fontweight='bold')

# Anotação destacando a diferença
ax.annotate(f'+{lift_raw} conversões\nna Variação B',
            xy=(1, cv_B), xytext=(1.15, cv_B - 60),
            color=GREEN, fontsize=10, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=GREEN))

ax.set_title('Total de Conversões por Grupo', color='white', fontsize=15, pad=15)
ax.set_ylabel('Conversões', color='white')
ax.set_ylim(0, max(valores_cv) * 1.2)
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_visible(False)

plt.tight_layout()
plt.savefig('fig2_conversoes.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Fig. 2 salva.')

In [ ]:
# --- GRÁFICO 3: Taxa de Conversão com IC 95% ---
# Este é o gráfico mais importante da análise.
# O intervalo de confiança (bigode) mostra a incerteza em torno de cada estimativa.
# Se os ICs NÃO se sobrepõem, há forte evidência de diferença real entre os grupos.
#
# Calculamos o IC de cada grupo individualmente usando a fórmula:
#   IC = CR ± Z * sqrt(CR*(1-CR)/n)   onde Z=1.96 para 95% de confiança

z_95 = 1.96
se_A = np.sqrt(cr_A * (1 - cr_A) / n_A)
se_B = np.sqrt(cr_B * (1 - cr_B) / n_B)

ic_A = (cr_A - z_95*se_A, cr_A + z_95*se_A)
ic_B = (cr_B - z_95*se_B, cr_B + z_95*se_B)

fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

crs  = [cr_A * 100, cr_B * 100]
errs = [z_95*se_A*100, z_95*se_B*100]

bars = ax.bar(grupos, crs, color=cores, width=0.5, alpha=0.9)
ax.errorbar(grupos, crs, yerr=errs, fmt='none', color='white',
            capsize=8, capthick=2, elinewidth=2)

for bar, val, ic_low, ic_high, err in zip(bars, crs,
    [ic_A[0]*100, ic_B[0]*100], [ic_A[1]*100, ic_B[1]*100], errs):
    # Percentual acima do IC
    ax.text(bar.get_x() + bar.get_width()/2, val + err + 0.4,
            f'{val:.2f}%', ha='center', va='bottom',
            color='white', fontsize=14, fontweight='bold')
    # IC abaixo da barra
    ax.text(bar.get_x() + bar.get_width()/2, 0.3,
            f'IC 95%: [{ic_low:.2f}%, {ic_high:.2f}%]',
            ha='center', va='bottom', color='#aaaaaa', fontsize=8)

# Anotação do lift
ax.annotate('', xy=(1, cr_B*100), xytext=(0, cr_A*100),
            arrowprops=dict(arrowstyle='<->', color=YELLOW, lw=2))
ax.text(0.5, (cr_A*100 + cr_B*100)/2 + 0.1,
        f'+{lift_pct*100:.1f}% lift\np = {2.8e-6:.7f}',
        ha='center', color=YELLOW, fontsize=9, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor=BG, edgecolor=YELLOW))

ax.set_title('Taxa de Conversão por Grupo (IC 95%)', color='white', fontsize=15, pad=15)
ax.set_ylabel('Taxa de Conversão (%)', color='white')
ax.set_ylim(0, max(crs) * 1.4)
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_visible(False)

plt.tight_layout()
plt.savefig('fig3_taxa_conversao.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Fig. 3 salva.')

## 5. Teste de Hipótese — Z-test para Proporções

### Por que o Z-test de proporções?

Temos **duas proporções independentes** (taxa de conversão do grupo A vs. grupo B) com amostras grandes (n > 1.000 por grupo). O Z-test é a escolha ideal porque:

1. **Escala:** Com n > 1.000 por grupo, o Teorema Central do Limite garante que a distribuição amostral das proporções é aproximadamente normal → Z-test válido
2. **Resultado direto:** Fornece Z-score, p-value e permite calcular o IC da *diferença* entre grupos
3. **Alternativas descartadas:**
   - *Qui-Quadrado*: matematicamente equivalente ao Z², mas não fornece IC da diferença diretamente
   - *Teste t*: adequado para médias contínuas, não para proporções
   - *Fisher Exato*: recomendado apenas para amostras pequenas (n < 30)

### Hipóteses

$$H_0: CR_B = CR_A \quad \text{(sem efeito — o headline não faz diferença)}$$
$$H_1: CR_B \neq CR_A \quad \text{(com efeito — bilateral, para capturar tanto melhora quanto piora)}$$

**Nível de significância:** α = 0,05 (padrão da indústria)

In [ ]:
# === CÁLCULO MANUAL (para entendimento pedagógico) ===
#
# Proporção pooled: usamos a proporção combinada dos dois grupos como
# estimativa da 'proporção verdadeira' sob H0 (que assume CR_A = CR_B)
p_pool = (cv_A + cv_B) / (n_A + n_B)

# Erro padrão da diferença
SE = np.sqrt(p_pool * (1 - p_pool) * (1/n_A + 1/n_B))

# Z-statistic: quantos desvios padrão a diferença observada está de zero
Z = (cr_B - cr_A) / SE

# P-value bilateral (two-tailed): probabilidade de observar um Z tão extremo
# se H0 fosse verdadeira
p_value = 2 * (1 - norm.cdf(abs(Z)))

# Intervalo de confiança de 95% para a DIFERENÇA entre os grupos
# Aqui usamos o erro padrão NÃO-pooled (para o IC, não assumimos H0)
se_diff = np.sqrt(cr_A*(1-cr_A)/n_A + cr_B*(1-cr_B)/n_B)
diff = cr_B - cr_A
ic_low  = diff - 1.96 * se_diff
ic_high = diff + 1.96 * se_diff

print('=== RESULTADOS DO Z-TEST ===')
print(f'\np̂_pool  = {p_pool:.5f}')
print(f'SE      = {SE:.6f}')
print(f'Z       = {Z:.4f}')
print(f'p-value = {p_value:.7f}')
print(f'\nIC 95% da diferença: [{ic_low*100:.2f} pp ; {ic_high*100:.2f} pp]')
print(f'Zero excluído do IC: {not (ic_low <= 0 <= ic_high)}')

alpha = 0.05
print(f'\n--- DECISÃO ---')
if p_value < alpha:
    print(f'✅ Rejeitamos H0 (p={p_value:.7f} < α={alpha})')
    print(f'   O resultado é estatisticamente significativo.')
else:
    print(f'❌ Não rejeitamos H0 (p={p_value:.4f} ≥ α={alpha})')

In [ ]:
# === VERIFICAÇÃO COM STATSMODELS ===
# Checamos os resultados manuais usando uma biblioteca consolidada

count = np.array([cv_B, cv_A])  # conversões [B, A]
nobs  = np.array([n_B,  n_A])   # totais    [B, A]

z_stat, p_val = proportions_ztest(count, nobs, alternative='two-sided')

print('=== VERIFICAÇÃO (statsmodels) ===')
print(f'Z-stat:  {z_stat:.4f}  (manual: {Z:.4f})  ✓' if abs(z_stat - Z) < 0.01 else f'Z-stat:  {z_stat:.4f}  (manual: {Z:.4f})  ⚠️')
print(f'p-value: {p_val:.7f}  (manual: {p_value:.7f})  ✓' if abs(p_val - p_value) < 1e-6 else f'p-value: {p_val:.7f}  ⚠️')

## 6. Análise de Poder Estatístico

### O que é e por que calcular?

O **poder estatístico** é a probabilidade de detectar um efeito real quando ele existe (= 1 − β, onde β é a taxa de erro Tipo II). Calculamos *post-hoc* (após o experimento) para:

1. **Validar a amostra:** Confirmar que n = 7.500/grupo foi suficiente para detectar um efeito do tamanho observado
2. **Comunicar confiança:** Um poder > 80% é o padrão mínimo aceitável na indústria

### Cohen h — Medida de Efeito para Proporções

Usamos **Cohen h** (e não diferença percentual simples) porque ele normaliza o tamanho do efeito levando em conta a natureza não-linear das proporções:

$$h = |2\arcsin(\sqrt{CR_B}) - 2\arcsin(\sqrt{CR_A})|$$

In [ ]:
# Cohen h: medida de efeito padronizada para comparação de duas proporções
# Usa transformação arco-seno para estabilizar a variância das proporções
effect_size = proportion_effectsize(cr_B, cr_A)
print(f'Cohen h = {effect_size:.4f}')
print(f'Classificação: {"pequeno" if abs(effect_size) < 0.2 else "médio" if abs(effect_size) < 0.5 else "grande"}'
      f' (referência: pequeno < 0.2, médio < 0.5, grande ≥ 0.5)')

# Tamanho mínimo de amostra necessário
# (para o efeito observado, com poder=80% e alpha=0.05)
analysis = NormalIndPower()
n_minimo = analysis.solve_power(
    effect_size=effect_size,
    alpha=0.05,
    power=0.80,
    alternative='two-sided'
)
print(f'\nTamanho mínimo por grupo (β=80%, α=0.05): {int(np.ceil(n_minimo)):,}')
print(f'Tamanho real (por grupo): ~{(n_A + n_B)//2:,}')
print(f'Razão (real/mínimo): {(n_A+n_B)/2 / n_minimo:.1f}×')

# Poder estatístico observado (post-hoc)
poder = analysis.solve_power(
    effect_size=effect_size,
    alpha=0.05,
    nobs1=n_A,
    alternative='two-sided'
)
print(f'\nPoder estatístico observado: {poder*100:.2f}%')
print(f'Mínimo aceitável: 80%  → {"✅ OK" if poder >= 0.8 else "❌ Insuficiente"}')

## 7. Validação por Bootstrap

### Por que fazer bootstrap além do Z-test?

O Z-test assume **distribuição normal** (via TCL). O bootstrap é uma técnica não-paramétrica que **não faz essa suposição** — ele estima a distribuição amostral empiricamente, reamostrando os dados com reposição.

Serve como **validação independente**: se o Z-test e o bootstrap concordam, temos muito mais confiança no resultado.

**Lógica:**
1. Reamostrar os dados do grupo A e B com reposição (simula "outro experimento")
2. Calcular o lift nessa reamostra
3. Repetir 10.000 vezes
4. Verificar em quantas % das simulações o lift foi positivo

In [ ]:
np.random.seed(42)  # Reprodutibilidade

n_bootstrap = 10_000

# Dados brutos de conversão por grupo
dados_A = df[df['variant']=='A']['converted'].values
dados_B = df[df['variant']=='B']['converted'].values

lifts_bootstrap = []

for _ in range(n_bootstrap):
    # Reamostra com reposição (mantendo o tamanho original do grupo)
    sample_A = np.random.choice(dados_A, size=len(dados_A), replace=True)
    sample_B = np.random.choice(dados_B, size=len(dados_B), replace=True)
    
    # Calcula o lift nessa reamostra
    cr_A_bs = sample_A.mean()
    cr_B_bs = sample_B.mean()
    lifts_bootstrap.append(cr_B_bs - cr_A_bs)

lifts_bootstrap = np.array(lifts_bootstrap)

# Proporção de reamostras com lift positivo (B > A)
pct_positivo = (lifts_bootstrap > 0).mean()

# IC de 95% do lift via percentis do bootstrap
ic_bs_low  = np.percentile(lifts_bootstrap, 2.5)
ic_bs_high = np.percentile(lifts_bootstrap, 97.5)

print('=== RESULTADOS DO BOOTSTRAP ===')
print(f'Iterações: {n_bootstrap:,}')
print(f'Lift mediano: +{np.median(lifts_bootstrap)*100:.2f} pp')
print(f'IC 95% (bootstrap): [{ic_bs_low*100:.2f} pp ; {ic_bs_high*100:.2f} pp]')
print(f'Reamostras com lift > 0: {pct_positivo*100:.1f}%')

print(f'\n--- COMPARAÇÃO ---')
print(f'IC 95% (Z-test):    [{ic_low*100:.2f} pp ; {ic_high*100:.2f} pp]')
print(f'IC 95% (Bootstrap): [{ic_bs_low*100:.2f} pp ; {ic_bs_high*100:.2f} pp]')
print(f'Concordância: ✅ Sim' if abs(ic_bs_low - ic_low) < 0.005 else '⚠️ Verificar')

In [ ]:
# Visualização da distribuição bootstrap
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

ax.hist(lifts_bootstrap * 100, bins=80, color=BLUE, alpha=0.8, edgecolor='none')
ax.axvline(0, color='red', lw=2, ls='--', label='Zero (H0)')
ax.axvline(np.median(lifts_bootstrap)*100, color=GREEN, lw=2, label=f'Mediana: +{np.median(lifts_bootstrap)*100:.2f} pp')
ax.axvline(ic_bs_low*100, color=YELLOW, lw=1.5, ls=':', label=f'IC 95%: [{ic_bs_low*100:.2f}, {ic_bs_high*100:.2f}] pp')
ax.axvline(ic_bs_high*100, color=YELLOW, lw=1.5, ls=':')

ax.set_title(f'Distribuição Bootstrap do Lift (n={n_bootstrap:,} reamostras)', color='white', fontsize=13)
ax.set_xlabel('Lift (pp)', color='white')
ax.set_ylabel('Frequência', color='white')
ax.tick_params(colors='white')
ax.legend(facecolor='#2a2a4a', labelcolor='white', fontsize=9)
for spine in ax.spines.values(): spine.set_visible(False)

plt.tight_layout()
plt.savefig('fig_bootstrap.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Fig. Bootstrap salva.')

## 8. Resumo Executivo

Consolidamos todos os resultados numa tabela única para facilitar a comunicação com stakeholders.

In [ ]:
print('=' * 55)
print(' RESUMO EXECUTIVO — A/B TEST SAASIFY')
print('=' * 55)
print(f"\n{'Métrica':<35} {'Valor':>18}")
print('-' * 55)
print(f"{'Grupo A — Taxa de Conversão':<35} {cr_A*100:>17.2f}%")
print(f"{'Grupo B — Taxa de Conversão':<35} {cr_B*100:>17.2f}%")
print(f"{'Lift Absoluto':<35} {lift_abs*100:>16.2f} pp")
print(f"{'Lift Percentual':<35} {lift_pct*100:>16.2f}%")
print(f"{'Trials Incrementais (14 dias)':<35} {lift_raw:>17,}")
print('-' * 55)
print(f"{'Z-Statistic':<35} {Z:>18.4f}")
print(f"{'p-value':<35} {p_value:>18.7f}")
print(f"{'IC 95% da Diferença':<35} [{ic_low*100:.2f}, {ic_high*100:.2f}] pp")
print(f"{'Significância (α=0.05)':<35} {'✅ SIM' if p_value < 0.05 else '❌ NÃO':>18}")
print('-' * 55)
print(f"{'Cohen h (effect size)':<35} {effect_size:>18.4f}")
print(f"{'Poder Estatístico':<35} {poder*100:>17.2f}%")
print(f"{'Bootstrap (lift > 0)':<35} {pct_positivo*100:>17.1f}%")
print('=' * 55)
print(f"\n🏆 VEREDICTO: Implementar a Variação B")
print(f"   Headline emocional converte +{lift_pct*100:.1f}% a mais.")

## 9. Próximos Passos

Mesmo com resultado positivo, recomendamos monitorar **métricas secundárias** nas próximas 4 semanas:

| Métrica | Por que monitorar |
|---|---|
| **Trial → Pago** | Mensagem emocional pode atrair usuários com menor intenção de pagar |
| **Ativação (D7)** | Usuários engajados nos primeiros 7 dias têm maior retenção |
| **Churn (30 dias)** | Garantir que o aumento de trials se converte em receita real |

**Plano de rollout sugerido:**
1. Feature flag: 20% → 50% → 100%
2. Monitorar métricas secundárias a cada etapa
3. Expandir a mensagem emocional para Ads e email nurturing

---
*Análise conduzida por **Victor Luhan** · Marketing Analytics Pleno · Python · SciPy · Statsmodels · Matplotlib*